# Risk-Adjusted Business Deal Valuation: Two-Stage Heavy-Tail Pipeline
### Dataset Structure: 1400 rows x 15 columns (orders.csv)

---

## Part 1: Briefing for Colleagues — Executive Data Quality & Generation Observations

### Data-Quality and Sparsity Diagnostics
* **`loss_amount_usd` (38 Missing):** This column is highly sparse and heavy-tailed. The 38 missing rows where `loss_recorded == 1` represent an operational logging leak. We drop these 38 rows to preserve target integrity.
* **`days_to_loss_recorded` (1190 Missing):** This missingness is **structural**, not an error. The values are empty precisely because no loss occurred (`loss_recorded == 0`). Including this as an input feature for classification induces an immediate **data leak** (the model reads the presence of a number as proof a loss will happen). It must be excluded from Phase 1 and used exclusively in Phase 2 conditionally.
* **`notes` (1241 Missing):** Free-form string fields break downstream models. We map this into a binary indicator feature: `has_notes` (0 or 1).
* **`buyer_region` (73 Missing):** Randomly unrecorded categorical feature. We fill these missing spaces with a distinct `'Unknown'` bucket to avoid geographic imputation bias.

### Synthetic Data Artifacts & Trust Vector
The uniform layout of missing records strongly indicates this dataset was simulated using a classic conditional mask (e.g., $P(\text{Missing} \mid \text{Loss}=0) = 1.0$). While functional for constructing pipelines, synthetic tail distributions often fail to mimic true macroeconomic correlation clusters. Our model might over-index on isolated features like `route_risk_flag` or `rep_confidence_score` without learning real-world systemic shock patterns.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, auc, f1_score, classification_report, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, HuberRegressor
import xgboost as xgb
import os

import warnings
warnings.filterwarnings('ignore')

print("All standard financial risk libraries successfully imported.")

All standard financial risk libraries successfully imported.


### Data Pipeline Setup
We attempt to load `orders.csv`. If it is absent from the directory, we programmatically construct a **1400x15 structural matrix** to ensure the code executes cell-by-cell without throwing errors.

In [4]:
if os.path.exists('dataset/orders.csv'):
    print("Loading existing orders.csv dataset...")
    df_raw = pd.read_csv('dataset/orders.csv')
else:
    print("orders.csv not found in working directory. Generating representative 1400x15 schema for pipeline integrity...")
    np.random.seed(42)
    n_samples = 1400
    
    # Creating the 15 columns matching your core metadata layout
    data = {
        'order_id': [f"ORD_{i:04d}" for i in range(n_samples)],
        'ship_date': pd.date_range(start='2024-01-01', periods=n_samples, freq='h'),
        'buyer_region': np.random.choice(['North', 'South', 'East', 'West', np.nan], size=n_samples, p=[0.4, 0.3, 0.15, 0.1, 0.05]),
        'product_category': np.random.choice(['Electronics', 'Industrial', 'Chemicals', 'Textiles'], size=n_samples),
        'order_value_usd': np.random.exponential(scale=15000, size=n_samples) + 2000,
        'buyer_tenure_yrs': np.random.randint(0, 12, size=n_samples),
        'new_buyer': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]),
        'payment_terms_days': np.random.choice([15, 30, 45, 60], size=n_samples),
        'route_risk_flag': np.random.choice([0, 1], size=n_samples, p=[0.92, 0.08]),
        'rep_confidence_score': np.random.uniform(50, 100, size=n_samples),
        'loss_recorded': 0, # Initializer
        'loss_amount_usd': np.nan, # Initializer
        'days_to_loss_recorded': np.nan, # Initializer
        'notes': np.nan,
        'extra_metric_dummy': np.random.randn(n_samples) # Pad out to exact column counts
    }
    df_raw = pd.DataFrame(data)
    
    # Injecting accurate historical distributions
    # We need loss_recorded to be 1 for a set of values
    loss_indices = df_raw.sample(n=210, random_state=42).index
    df_raw.loc[loss_indices, 'loss_recorded'] = 1
    
    # Setting up heavy-tailed losses (Pareto distribution)
    base_loss = (np.random.pareto(a=1.1, size=len(loss_indices)) + 1) * 12000
    df_raw.loc[loss_indices, 'loss_amount_usd'] = base_loss
    
    # Injecting target corruptions (38 rows missing target amounts)
    corrupt_idx = pd.Series(loss_indices).sample(n=38, random_state=42).values
    df_raw.loc[corrupt_idx, 'loss_amount_usd'] = np.nan
    
    # Structural missingness parameters
    valid_loss_idx = df_raw[(df_raw['loss_recorded'] == 1) & (df_raw['loss_amount_usd'].notnull())].index
    df_raw.loc[valid_loss_idx, 'days_to_loss_recorded'] = np.random.randint(5, 45, size=len(valid_loss_idx))
    
    # Fill Notes fields
    note_idx = df_raw.sample(n=159, random_state=42).index
    df_raw.loc[note_idx, 'notes'] = "Risk discrepancy or delivery variance flagged."

print("Shape of dataset in pipe:", df_raw.shape)
print(df_raw.isnull().sum())

Loading existing orders.csv dataset...
Shape of dataset in pipe: (1400, 15)
order_id                    0
ship_date                   0
buyer_region               73
product_category            0
order_value_usd             0
buyer_tenure_yrs            0
new_buyer                   0
payment_terms_days          0
route_risk_flag             0
rep_confidence_score        0
loss_recorded               0
loss_type                   0
loss_amount_usd            38
days_to_loss_recorded    1190
notes                    1241
dtype: int64


### Data Cleansing Framework
We now clean the schema deterministically based on our strategy outline.

In [5]:
# 1. Drop corrupted target instances
corrupted_condition = (df_raw['loss_recorded'] == 1) & (df_raw['loss_amount_usd'].isnull())
df_clean = df_raw[~corrupted_condition].copy()

# 2. Transform unstructured notes fields into binary indicators
df_clean['has_notes'] = df_clean['notes'].notnull().astype(int)
df_clean.drop(columns=['notes'], inplace=True, errors='ignore')

# 3. Unify undefined regional factors
df_clean['buyer_region'] = df_clean['buyer_region'].fillna('Unknown')

print("Cleaned Dataset Structural State:", df_clean.shape)
print(df_clean.isnull().sum())

Cleaned Dataset Structural State: (1362, 15)
order_id                    0
ship_date                   0
buyer_region                0
product_category            0
order_value_usd             0
buyer_tenure_yrs            0
new_buyer                   0
payment_terms_days          0
route_risk_flag             0
rep_confidence_score        0
loss_recorded               0
loss_type                   0
loss_amount_usd             0
days_to_loss_recorded    1190
has_notes                   0
dtype: int64


In [7]:
df_clean.head()

,order_id,ship_date,buyer_region,product_category,order_value_usd,buyer_tenure_yrs,new_buyer,payment_terms_days,route_risk_flag,rep_confidence_score,loss_recorded,loss_type,loss_amount_usd,days_to_loss_recorded,has_notes
0,18,2025-01-17,EU-West,beauty,4279.91,5.2,0,30,medium,0.86,0,none,0.00,NaN,0
1,99,2025-05-26,EU-West,home_appliance,14589.12,10.7,0,0,medium,0.49,1,dispute,759.70,79.0,1
2,790,2024-01-10,EU-North,pet_care,3918.59,2.7,0,90,medium,0.32,1,return,907.74,29.0,0
3,631,2024-03-15,EU-South,home_appliance,12236.41,7.3,0,30,low,0.39,0,none,0.00,NaN,0
4,811,2025-04-14,EU-West,pet_care,5810.34,5.1,0,30,low,0.75,0,none,0.00,NaN,0


## Part 2: Two-Stage Hurdle Model Compilation

### Modeling Choice & Justification
A direct single-stage regression model targeting `loss_amount_usd` will entirely fail because over 80% of the dataset features a true loss of $0. The model would aggressively skew towards zero, underestimating severe exposures.

**Our Solution:** A **Two-Stage Hurdle Framework**:
1. **Stage 1 (Classification Model):** Estimates $P(\text{Loss}=1 \mid X)$. Evaluated using **PR-AUC** to protect the company against severe class imbalances.
2. **Stage 2 (Heavy-Tail Robust Regression Model):** Trained *only* on sub-segments where a loss was realized. We employ log-transformed labels optimized through robust algorithms like **Huber Loss** to prevent severe outliers from distorting convergence parameters.

In [25]:
# One-Hot Encoding
categorical_cols = ['buyer_region', 'product_category']
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

# Ensure leak targets are fully stripped from features
exclude_features = ['order_id', 'ship_date', 'loss_recorded', 'loss_amount_usd', 'days_to_loss_recorded']
feature_cols = [c for c in df_encoded.columns if c not in exclude_features]

X = df_encoded[feature_cols]
y_class = df_encoded['loss_recorded']

X_train, X_test, y_train_cls, y_test_cls = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)

print(f"Training set shape: {X_train.shape}, Imbalance distribution: {np.bincount(y_train_cls)}")

Training set shape: (1089, 15), Imbalance distribution: [951 138]


In [46]:
print (df_clean['product_category'].unique())

['beauty' 'home_appliance' 'pet_care' 'cleaning']


In [48]:
# 1. Inspect which columns contain string data that should be numeric
print("Current column data types:")
print(df_clean.dtypes)

# 2. Fix Ordinal Strings (e.g., 'low', 'medium', 'high') to numeric values
# Identify the column causing the error (replace 'route_risk_flag' with your actual column if different)
target_string_column = 'route_risk_flag' 

if target_string_column in df_clean.columns and df_clean[target_string_column].dtype == 'object':
    print(f"Mapping text levels in '{target_string_column}' to numerical values...")
    
    # Create an explicit mapping dictionary for ordinal risk scales
    risk_mapping = {
        'low': 1,
        'medium': 2,
        'high': 3
    }
    
    # Apply the mapping and fill any unmapped strings with a default neutral median (e.g., 2)
    df_clean[target_string_column] = df_clean[target_string_column].astype(str).str.lower().map(risk_mapping).fillna(2)

# 3. Clean any accidental string representations of numbers (e.g., '12.5' stored as string)
# Force all remaining features in your matrix (excluding ID and dates) to numeric, converting errors to NaN
for col in feature_cols:
    if col in df_encoded.columns and df_encoded[col].dtype == 'object':
        # If it cannot be parsed into a float, it becomes NaN, then we fill with 0
        df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce').fillna(0)

print("Data type conversion complete. Ready for model training.")


Current column data types:
order_id                   int64
ship_date                 object
buyer_region              object
product_category          object
order_value_usd          float64
buyer_tenure_yrs         float64
new_buyer                  int64
payment_terms_days         int64
route_risk_flag            int64
rep_confidence_score     float64
loss_recorded              int64
loss_type                  int64
loss_amount_usd          float64
days_to_loss_recorded    float64
has_notes                  int32
dtype: object
Data type conversion complete. Ready for model training.


In [49]:
# 1. Inspect which columns contain string data that should be numeric
print("Current column data types:")
print(df_clean.dtypes)

# 2. Fix Ordinal Strings (e.g., 'low', 'medium', 'high') to numeric values
# Identify the column causing the error (replace 'route_risk_flag' with your actual column if different)
target_string_column = 'loss_type' 

if target_string_column in df_clean.columns and df_clean[target_string_column].dtype == 'object':
    print(f"Mapping text levels in '{target_string_column}' to numerical values...")
    
    # Create an explicit mapping dictionary for ordinal risk scales
    risk_mapping = {
        'none': 0,
        'dispute': 1,
        'return': 2,
        'freight_overrun': 3,
        'late_payment': 4,
        'default': 5
    }
    
    # Apply the mapping and fill any unmapped strings with a default neutral median (e.g., 2)
    df_clean[target_string_column] = df_clean[target_string_column].astype(str).str.lower().map(risk_mapping).fillna(2)

# 3. Clean any accidental string representations of numbers (e.g., '12.5' stored as string)
# Force all remaining features in your matrix (excluding ID and dates) to numeric, converting errors to NaN
for col in feature_cols:
    if col in df_encoded.columns and df_encoded[col].dtype == 'object':
        # If it cannot be parsed into a float, it becomes NaN, then we fill with 0
        df_encoded[col] = pd.to_numeric(df_encoded[col], errors='coerce').fillna(0)

print("Data type conversion complete. Ready for model training.")


Current column data types:
order_id                   int64
ship_date                 object
buyer_region              object
product_category          object
order_value_usd          float64
buyer_tenure_yrs         float64
new_buyer                  int64
payment_terms_days         int64
route_risk_flag            int64
rep_confidence_score     float64
loss_recorded              int64
loss_type                  int64
loss_amount_usd          float64
days_to_loss_recorded    float64
has_notes                  int32
dtype: object
Data type conversion complete. Ready for model training.


### Stage 1: Evaluating Multiple Classifiers via PR-AUC and F1-Score

In [54]:
models_cls = {
    "Logistic Regression (Balanced)": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost Classifier": xgb.XGBClassifier(scale_pos_weight=(len(y_train_cls)-sum(y_train_cls))/sum(y_train_cls), eval_metric='logloss', random_state=42)
}

for name, model in models_cls.items():
    model.fit(X_train, y_train_cls)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    precision, recall, _ = precision_recall_curve(y_test_cls, probs)
    pr_auc = auc(recall, precision)
    f1 = f1_score(y_test_cls, preds)
    
    print(f"=== {name} ===")
    print(f"PR-AUC Score: {pr_auc:.4f} | Balanced F1-Score: {f1:.4f}")
    print(classification_report(y_test_cls, preds))
    print("\n")

ValueError: could not convert string to float: 'none'

### Stage 2: Heavy-Tail Severity Extrapolation Regression

In [15]:
# Filter down strictly to active loss instances
df_losses = df_encoded[df_encoded['loss_recorded'] == 1]
X_reg = df_losses[feature_cols]
y_reg = df_losses['loss_amount_usd']

# Mitigate extreme variance via natural logarithmic compression
y_reg_log = np.log1p(y_reg)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg_log, test_size=0.2, random_state=42)

models_reg = {
    "Huber Robust Regressor": HuberRegressor(max_iter=2000)
}

for name, model in models_reg.items():
    model.fit(X_train_reg, y_train_reg)
    log_preds = model.predict(X_test_reg)
    
    # Inverse transform to compute metrics in true USD terms
    true_preds = np.expm1(log_preds)
    true_target = np.expm1(y_test_reg)
    
    mae = mean_absolute_error(true_target, true_preds)
    r2 = r2_score(true_target, true_preds)
    
    print(f"=== {name} ===")
    print(f"True Dollar Metric Mean Absolute Error: ${mae:,.2f}")
    print(f"R2 Valuation Stability Metric: {r2:.4f}\n")

=== Huber Robust Regressor ===
True Dollar Metric Mean Absolute Error: $849.46
R2 Valuation Stability Metric: -0.0448

